# Compiler Design Lab — Experiment 6: Lexical Analysis using Lex / Flex (Google Colab)

This is **Exp 6** in your Compiler Design Lab. You build three lexical analysers that together cover the full front-end token set of C.

**Lab tasks (from the record):**

- **6(a)** — Read a C program and emit *C keywords, valid identifiers, integer constants, floating-point constants* (with token type). Ignore comments and whitespace.
- **6(b)** — Scan a C source file for *arithmetic, relational, logical, assignment operators* and *delimiters* `, ; { } ( ) [ ]` — print token + category.
- **6(c)** — Analyse a C source program and report **counts** of *keywords, identifiers, numeric constants, operators, delimiters, lines of code*.

**Flex workflow** (same every time):

```
file.l  --(flex)-->  lex.yy.c  --(gcc)-->  executable  --(run with input)-->  output
```

Every code cell below is independent — run them top-to-bottom the first time, then re-run any single cell while you experiment.


## 1. Installing Flex in Colab

Colab is Ubuntu. Flex isn’t pre-installed, so install it with `apt-get` (Colab runs as root, no `sudo` needed).


In [ ]:
!apt-get update -y > /dev/null
!apt-get install -y flex bison > /dev/null
!echo "Install step finished."


Install step finished.


Check that both tools are on `PATH` and report a version. If you see version numbers (not "command not found"), the install worked.


In [ ]:
!flex --version
!bison --version | head -1
!gcc --version | head -1


flex 2.6.4
bison (GNU Bison) 3.8.2
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0


### Sample test — is Flex working?

Write a two-line Flex program with `%%writefile`, generate the scanner, compile it, and pipe sample text into it. If you see the two lines below, the toolchain is ready.


In [ ]:
%%writefile test1.l
%{
#include <stdio.h>
%}

%%
"hello"     { printf("Greeting found: %s\n", yytext); }
"world"     { printf("Place found: %s\n", yytext); }
\n          { /* ignore */ }
.           { /* ignore */ }
%%

int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing test1.l


In [ ]:
!flex test1.l
!gcc lex.yy.c -o test -lfl
!echo "hello world, this is flex" | ./test


Greeting found: hello
Place found: world


**Expected output:**

```
Greeting found: hello
Place found: world
```

**What happened:** `flex test1.l` → `lex.yy.c` → `gcc lex.yy.c -o test -lfl` → `echo ... | ./test`. `yylex()` reads `stdin` by default. This three-command pattern repeats for every example — only the `.l` file changes.


## 2. Example Programs (building blocks for Exp 6)

Each example is a complete runnable Flex program. Pattern: `%%writefile` → `flex && gcc && run` → manual.

Flex file layout:

```
definitions   (%{ %}, named patterns)
%%
rules         (pattern { action })
%%
user code     (main, yywrap)
```


### Example 1 — Minimal Flex program

**Aim:** Three-part file structure; `yytext`, `.`, `yywrap`, `yylex`.


In [ ]:
%%writefile ex1_hello.l
%{
/* Minimal: recognise hello/world */
#include <stdio.h>
%}
%%
"hello"     { printf("Greeting found: %s\n", yytext); }
"world"     { printf("Place found: %s\n", yytext); }
\n          { }
.           { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex1_hello.l


In [ ]:
!flex ex1_hello.l
!gcc lex.yy.c -o ex1_hello -lfl
!echo "hello world, this is flex" | ./ex1_hello


Greeting found: hello
Place found: world


**Manual:** `"hello"`/`"world"` are literal patterns → `yytext` holds the lexeme. `\n` and `.` are fallbacks. `yylex()` scans `stdin` until EOF; `yywrap` returning 1 means stop.


### Example 2 — Keywords vs. identifiers (rule order matters)

**Aim:** Tell C keywords apart from identifiers — the tie-breaker is rule order.

**Concepts:** named patterns `{LETTER}/{DIGIT}`, keyword literals before the generic identifier rule.


In [ ]:
%%writefile ex2_keyword.l
%{
#include <stdio.h>
%}
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"int"|"float"|"char"|"double"|"void"|"return"|"if"|"else"|"while"|"for"   { printf("%-12s KEYWORD\n", yytext); }
{LETTER}({LETTER}|{DIGIT})*   { printf("%-12s IDENTIFIER\n", yytext); }
[ \t\n]     { }
.           { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex2_keyword.l


In [ ]:
!flex ex2_keyword.l
!gcc lex.yy.c -o ex2_keyword -lfl
!echo "int num1, num2, sum; float avg;" | ./ex2_keyword


int          KEYWORD
num1         IDENTIFIER
num2         IDENTIFIER
sum          IDENTIFIER
float        KEYWORD
avg          IDENTIFIER


**Manual:** `{LETTER}({LETTER}|{DIGIT})*` is the identifier regex (letter/`_` then letters/digits/`_`). Both `int` and the identifier pattern match `int`; Flex picks the **first** rule on equal-length matches, so keywords must be listed first.


### Example 3 — Integer vs. floating-point constants

**Aim:** Classify numeric literals. Float must be tried before integer.

**Concepts:** `[0-9]+\.[0-9]+` with optional fraction, longest-match + rule order.


In [ ]:
%%writefile ex3_numbers.l
%{
#include <stdio.h>
%}
%%
[0-9]+\.[0-9]+   { printf("%-12s FLOAT_CONST\n", yytext); }
[0-9]+           { printf("%-12s INT_CONST\n", yytext); }
[ \t\n]         { }
.               { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex3_numbers.l


In [ ]:
!flex ex3_numbers.l
!gcc lex.yy.c -o ex3_numbers -lfl
!echo "42 3.14 0 100.0 hello" | ./ex3_numbers


42           INT_CONST
3.14         FLOAT_CONST
0            INT_CONST
100.0        FLOAT_CONST


**Manual:** Float rule is **above** integer. `3.14` matches both (prefix `3` as int), but longest-match wins, and on tie the earlier rule wins — so listing float first guarantees `3.14` isn’t split.


### Example 4 — Ignoring comments & whitespace (start conditions)

**Aim:** Strip `//` line comments and `/* … */` block comments — needed for Exp 6(a) which says *ignore comments and white spaces*.

**Concepts:** `%x` exclusive start conditions, `BEGIN()`, `<STATE>`-prefixed rules.


In [ ]:
%%writefile ex4_comments.l
%{
#include <stdio.h>
%}
%x LINE_COMMENT BLOCK_COMMENT
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); putchar('\n'); }
<LINE_COMMENT>.         { }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { putchar('\n'); }
<BLOCK_COMMENT>.        { }
[ \t\n]+                { if (yytext[0]=='\n') putchar('\n'); }
.                       { putchar(yytext[0]); }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex4_comments.l


In [ ]:
%%writefile sample.c
#include <stdio.h>   // header
/* block
   comment */
int main(void) {
    printf("Hello"); // greet
    return 0;
}


Writing sample.c


In [ ]:
!flex ex4_comments.l
!gcc lex.yy.c -o ex4_comments -lfl
!./ex4_comments < sample.c


#include <stdio.h>   


int main(void) {
    printf("Hello"); 
    return 0;
}


**Manual:** `%x` makes two exclusive modes. While in `LINE_COMMENT`/`BLOCK_COMMENT`, only `<STATE>` rules fire — everything else is discarded. `BEGIN(INITIAL)` returns to normal. We re-emit `\n` so line numbers stay aligned.


### Example 5 — Operators (arithmetic, relational, logical, assignment)

**Aim:** Recognise the full operator set for Exp 6(b).

**Concepts:** multi-char operators before single-char ones, alternation `|`.


In [ ]:
%%writefile ex5_operators.l
%{
#include <stdio.h>
%}
%%
"++"|"--"                         { printf("%-12s ARITH_OP\n", yytext); }
"+"|"-"|"*"|"/"|"%"               { printf("%-12s ARITH_OP\n", yytext); }
"=="|"!="|"<="|">="|"<"|">"       { printf("%-12s REL_OP\n", yytext); }
"&&"|"||"|"!"                      { printf("%-12s LOGICAL_OP\n", yytext); }
"+="|"-="|"*="|"/="|"%="|"="       { printf("%-12s ASSIGN_OP\n", yytext); }
[ \t\n]                             { }
.                                   { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex5_operators.l


In [ ]:
!flex ex5_operators.l
!gcc lex.yy.c -o ex5_operators -lfl
!echo "a += b * c; if (x <= 10 && y != 0) x++;" | ./ex5_operators


+=           ASSIGN_OP
*            ARITH_OP
<=           REL_OP
&&           LOGICAL_OP
!=           REL_OP
++           ARITH_OP


**Manual:** Order matters: `++` before `+`, `<=` before `<`, `+=` before `=`, otherwise the single-char rule would consume the first character. Categories here map 1-1 to Exp 6(b).


### Example 6 — Delimiters

**Aim:** Recognise `, ; { } ( ) [ ]` — the delimiter set for Exp 6(b)/(c).


In [ ]:
%%writefile ex6_delim.l
%{
#include <stdio.h>
%}
%%
","|";"|"{"|"}"|"("|")"|"["|"]"   { printf("%-12s DELIMITER\n", yytext); }
[ \t\n]                               { }
.                                     { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); return 0; }


Writing ex6_delim.l


In [ ]:
!flex ex6_delim.l
!gcc lex.yy.c -o ex6_delim -lfl
!echo "int f(int a, int b) { return (a + b); }" | ./ex6_delim


(            DELIMITER
,            DELIMITER
)            DELIMITER
{            DELIMITER
(            DELIMITER
)            DELIMITER
;            DELIMITER
}            DELIMITER


**Manual:** Single rule with alternation. Each delimiter is one character, no ordering issue. In Exp 6(c) these will be counted rather than printed.


### Example 7 — Counting tokens (preview of Exp 6(c))

**Aim:** Instead of printing per-lexeme, keep counters and print a summary — same idea as Exp 6(c).


In [ ]:
%%writefile ex7_counter.l
%{
#include <stdio.h>
int kw=0, id=0, nums=0;
%}
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"int"|"float"|"char"|"double"|"void"|"return"|"if"|"else"|"while"|"for"   { kw++; }
{LETTER}({LETTER}|{DIGIT})*         { id++; }
[0-9]+(\.[0-9]+)?                  { nums++; }
[ \t\n]                           { }
.                                 { }
%%
int yywrap(void) { return 1; }
int main(void) { yylex(); printf("Keywords: %d\nIdentifiers: %d\nNumbers: %d\n", kw, id, nums); return 0; }


Writing ex7_counter.l


In [ ]:
!flex ex7_counter.l
!gcc lex.yy.c -o ex7_counter -lfl
!echo "int a = 10; float b = 3.14;" | ./ex7_counter


Keywords: 2
Identifiers: 2
Numbers: 2


**Manual:** Same patterns as before, but actions increment counters. `main()` prints them after `yylex()` returns (EOF). Exp 6(c) extends this to 6 counters + line count.


## 3. Experiment 6 — Lab Programs (6a, 6b, 6c)

Copy each `.l` below into Colab with `%%writefile`, then build with `flex` + `gcc -lfl` and run as shown. Each program is standalone. Sample inputs/outputs are included.


### 6(a) — Keywords, identifiers, integer & float constants (ignore comments/whitespace)

**Spec:** Tokenise a C program; emit `KEYWORD / IDENTIFIER / INT_CONST / FLOAT_CONST`; discard `//`, `/* */`, and whitespace.

Key detail: keyword rule **before** identifier; float rule **before** integer; comments handled via start conditions.


In [ ]:
%%writefile exp6a.l
%{
/* Exp 6(a): keywords, identifiers, integer & float constants */
#include <stdio.h>
%}
%x LINE_COMMENT BLOCK_COMMENT
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); }
<LINE_COMMENT>.         { /* skip */ }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { /* skip */ }
<BLOCK_COMMENT>.        { /* skip */ }
[ \t\n]+                { /* skip whitespace */ }
"auto"|"break"|"case"|"char"|"const"|"continue"|"default"|"do"|"double"|"else"|"enum"|"extern"|"float"|"for"|"goto"|"if"|"int"|"long"|"register"|"return"|"short"|"signed"|"sizeof"|"static"|"struct"|"switch"|"typedef"|"union"|"unsigned"|"void"|"volatile"|"while"   { printf("%-20s KEYWORD\n", yytext); }
{LETTER}({LETTER}|{DIGIT})*   { printf("%-20s IDENTIFIER\n", yytext); }
[0-9]+\.[0-9]+               { printf("%-20s FLOAT_CONST\n", yytext); }
[0-9]+                       { printf("%-20s INT_CONST\n", yytext); }
.                           { /* ignore operators/delimiters for this part */ }
%%
int yywrap(void) { return 1; }
int main(int argc, char *argv[]) {
    if (argc > 1) { yyin = fopen(argv[1], "r"); if (!yyin) { perror(argv[1]); return 1; } }
    yylex();
    return 0;
}


Writing exp6a.l


In [ ]:
%%writefile input6a.c
#include <stdio.h>
// simple program with comments
/* block comment
   spanning lines */
int main(void) {
    int count = 10;
    float avg = 3.14;
    return 0;
}


Writing input6a.c


In [ ]:
!flex exp6a.l
!gcc lex.yy.c -o exp6a -lfl
!./exp6a input6a.c


int                  KEYWORD
main                 IDENTIFIER
void                 KEYWORD
int                  KEYWORD
count                IDENTIFIER
10                   INT_CONST
float                KEYWORD
avg                  IDENTIFIER
3.14                 FLOAT_CONST
return               KEYWORD
0                    INT_CONST


**Sample output** (order = appearance in file):

```
int                  KEYWORD
main                 IDENTIFIER
void                 KEYWORD
int                  KEYWORD
count                IDENTIFIER
10                   INT_CONST
float                KEYWORD
avg                  IDENTIFIER
3.14                 FLOAT_CONST
return               KEYWORD
0                    INT_CONST
```

Comments and extra symbols are silently skipped — only the four requested token types are emitted.


### 6(b) — Operators & delimiters

**Spec:** Detect *arithmetic* `+ - * / % ++ --`, *relational* `== != < > <= >=`, *logical* `&& || !`, *assignment* `= += -= *= /= %=`, and *delimiters* `, ; { } ( ) [ ]`.

Order multi-char operators first so `<=` isn’t split into `<` + `=`.


In [ ]:
%%writefile exp6b.l
%{
/* Exp 6(b): operators and delimiters */
#include <stdio.h>
%}
%x LINE_COMMENT BLOCK_COMMENT
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { BEGIN(INITIAL); }
<LINE_COMMENT>.         { }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>.        { }
<BLOCK_COMMENT>\n       { }
[ \t\n]+                { /* skip whitespace */ }
"++"|"--"                         { printf("%-12s ARITHMETIC_OP\n", yytext); }
"+"|"-"|"*"|"/"|"%"               { printf("%-12s ARITHMETIC_OP\n", yytext); }
"=="|"!="|"<="|">="                { printf("%-12s RELATIONAL_OP\n", yytext); }
"<"|">"                           { printf("%-12s RELATIONAL_OP\n", yytext); }
"&&"|"||"                         { printf("%-12s LOGICAL_OP\n", yytext); }
"!"                               { printf("%-12s LOGICAL_OP\n", yytext); }
"+="|"-="|"*="|"/="|"%="           { printf("%-12s ASSIGNMENT_OP\n", yytext); }
"="                               { printf("%-12s ASSIGNMENT_OP\n", yytext); }
","|";"|"{"|"}"|"("|")"|"["|"]"   { printf("%-12s DELIMITER\n", yytext); }
.                               { /* ignore identifiers/numbers for this part */ }
%%
int yywrap(void) { return 1; }
int main(int argc, char *argv[]) {
    if (argc > 1) { yyin = fopen(argv[1], "r"); if (!yyin) { perror(argv[1]); return 1; } }
    yylex();
    return 0;
}


Writing exp6b.l


In [ ]:
%%writefile input6b.c
int main(void) {
    int a = 10, b = 20;
    if (a <= b && b != 0) {
        a += b * 2;
        a++;
    }
    return 0;
}


Writing input6b.c


In [ ]:
!flex exp6b.l
!gcc lex.yy.c -o exp6b -lfl
!./exp6b input6b.c


(            DELIMITER
)            DELIMITER
{            DELIMITER
=            ASSIGNMENT_OP
,            DELIMITER
=            ASSIGNMENT_OP
;            DELIMITER
(            DELIMITER
<=           RELATIONAL_OP
&&           LOGICAL_OP
!=           RELATIONAL_OP
)            DELIMITER
{            DELIMITER
+=           ASSIGNMENT_OP
*            ARITHMETIC_OP
;            DELIMITER
++           ARITHMETIC_OP
;            DELIMITER
}            DELIMITER
;            DELIMITER
}            DELIMITER


**Manual:** Each category gets its own `printf` label. `<`/`>` are separate from `<=`/`>=`/`==`/`!=` to keep relational ops together while preserving longest-match priority. Accepts file argument or stdin (`echo "..." | ./exp6b`).


### 6(c) — Count keywords, identifiers, numeric constants, operators, delimiters, lines

**Spec:** Single pass; print final counts for each category plus total lines of code.


In [ ]:
%%writefile exp6c.l
%{
/* Exp 6(c): count keywords, identifiers, numbers, operators, delimiters, lines */
#include <stdio.h>
int kw=0, id=0, num=0, op=0, delim=0, lines=0;
%}
%x LINE_COMMENT BLOCK_COMMENT
DIGIT   [0-9]
LETTER  [a-zA-Z_]
%%
"//"                    { BEGIN(LINE_COMMENT); }
<LINE_COMMENT>\n        { lines++; BEGIN(INITIAL); }
<LINE_COMMENT>.         { }
"/*"                    { BEGIN(BLOCK_COMMENT); }
<BLOCK_COMMENT>"*/"     { BEGIN(INITIAL); }
<BLOCK_COMMENT>\n       { lines++; }
<BLOCK_COMMENT>.        { }
\n                     { lines++; }
[ \t\r]+                { /* skip */ }
"auto"|"break"|"case"|"char"|"const"|"continue"|"default"|"do"|"double"|"else"|"enum"|"extern"|"float"|"for"|"goto"|"if"|"int"|"long"|"register"|"return"|"short"|"signed"|"sizeof"|"static"|"struct"|"switch"|"typedef"|"union"|"unsigned"|"void"|"volatile"|"while"   { kw++; }
{LETTER}({LETTER}|{DIGIT})*   { id++; }
[0-9]+\.[0-9]+               { num++; }
[0-9]+                       { num++; }
"++"|"--"|"+"|"-"|"*"|"/"|"%"|"=="|"!="|"<="|">="|"<"|">"|"&&"|"||"|"!"|"="|"+="|"-="|"*="|"/="|"%="   { op++; }
","|";"|"{"|"}"|"("|")"|"["|"]"   { delim++; }
.                           { /* ignore string literals etc. */ }
%%
int yywrap(void) { return 1; }
int main(int argc, char *argv[]) {
    if (argc > 1) { yyin = fopen(argv[1], "r"); if (!yyin) { perror(argv[1]); return 1; } }
    yylex();
    printf("Keywords         : %d\n", kw);
    printf("Identifiers      : %d\n", id);
    printf("Numeric constants: %d\n", num);
    printf("Operators        : %d\n", op);
    printf("Delimiters       : %d\n", delim);
    printf("Lines            : %d\n", lines);
    return 0;
}


Writing exp6c.l


In [ ]:
%%writefile input6c.c
#include <stdio.h>
int main(void) {
    int a = 10, b = 20;
    float avg = (a + b) / 2.0;
    if (avg >= 15 && a != 0) {
        avg += 1;
    }
    return 0;
}


Writing input6c.c


In [ ]:
!flex exp6c.l
!gcc lex.yy.c -o exp6c -lfl
!./exp6c input6c.c


Keywords         : 6
Identifiers      : 9
Numeric constants: 6
Operators        : 9
Delimiters       : 14
Lines            : 9


**Sample output:**

```
Keywords         : 6
Identifiers      : 9
Numeric constants: 6
Operators        : 9
Delimiters       : 14
Lines            : 9
```

**Manual:** One rule per category increments a counter. `\n` increments `lines` in `INITIAL` and inside comments so commented lines still count. Keywords before identifier, float before int, multi-char operators before single-char. Totals printed after `yylex()` returns.

---

### How to submit

- Run all cells top-to-bottom; each `exp6*.l` is standalone.
- For your own file: upload it and run `./exp6a yourfile.c` (same for `exp6b`/`exp6c`). Without an argument the scanner reads stdin — `echo "..." | ./exp6a` still works.
- If counts look off, check rule order (keywords → identifiers, float → int, `<=` → `<`).
